# Agent Foundations and Tool Contracts

> **The story.** In 2022, reasoning-and-acting systems such as ReAct made language-model actions observable as alternating thoughts, tool calls, and observations. The useful boundary was not that a model could describe an action; it was that software could validate and execute a named action. OrderFlow begins at that boundary.
>
> **Where you are.** You already know how language models generate text from the GenAI track. CFO Elena Vasquez now needs PO `#7293` processed without letting prose become an untraceable financial action. This chapter builds the smallest useful agent before any framework hides the loop.
>
> **Notation.** $r$ is a purchase request; $a_t$ is the proposed action at step $t$; $o_t$ is the resulting observation; $T$ is the registered tool set; $V(a_t)$ is schema and policy validation.

## 0 - The Challenge

![A purchase-order email becomes proposed actions, passes registered-name, typed-argument, and policy validation, then produces an attributable inventory observation](../images/ch00-tool-contract-boundary.png)

> **The mission:** move PO `#7293` from free-form email to an attributable inventory and price check.

**What we know so far:**

- A model can describe what purchasing staff should do.
- **But prose cannot prove that a real tool exists or that its arguments are valid.**

**What's blocking us:** the naive model emits plausible text, wrong tool names, and incomplete arguments. OrderFlow's first target is 10/10 schema-valid calls on representative requests, with unknown SKUs rejected instead of invented.

```mermaid
flowchart LR
    A["PO email"] --> B["Model prose"]
    B --> C["Malformed or invented action"]
    C --> D["Typed tool contract"]
    D --> E["Validated observation"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Setup: deterministic OrderFlow fixtures -------------------------------
from pathlib import Path
import json
import re
import sys
from typing import Any, Callable

TRACK_DIR = Path(__vsc_ipynb_file__).resolve().parents[1] if '__vsc_ipynb_file__' in globals() else Path.cwd().resolve().parents[0]
if str(TRACK_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_DIR))

from pydantic import BaseModel, Field, ValidationError, field_validator
from shared import DeterministicModel, INVENTORY, load_purchase_requests

requests = [request for request in load_purchase_requests() if request['solvable']][:10]
model = DeterministicModel()

assert len(requests) == 10
print(f'Loaded {len(requests)} solvable OrderFlow requests; no API key or network required.')

## 1 - The Agency Boundary: A Plan Is Not an Action

**Predict:** The fake model returns text that looks reasonable. Which failure appears first: invalid JSON, an unregistered tool name, or missing arguments?

```mermaid
flowchart TD
    R["Request"] --> M["Model proposes"]
    M --> Q{"Can software validate it?"}
    Q -->|"No"| F["Stop: description only"]
    Q -->|"Yes"| X["Execute registered tool"]
    style R fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Q fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Failure first: parse plausible prose as if it were a tool call --------
def naive_parse(text: str) -> dict[str, Any] | None:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

naive_results = []
for request in requests:
    proposal = model.plain_plan(request['email'])
    parsed = naive_parse(proposal)
    valid = bool(
        parsed
        and parsed.get('name') == 'check_inventory'
        and set(parsed.get('arguments', {})) == {'sku', 'quantity'}
    )
    naive_results.append(valid)

naive_rate = sum(naive_results) / len(naive_results)
print(f'Naive valid-call rate: {sum(naive_results)}/{len(naive_results)} ({naive_rate:.0%})')
print('Failure observed: plausible language is not an executable contract.')
assert naive_rate < 1.0

The model did not become an agent when it mentioned a tool. Agency begins where a validated action can cross a software boundary. The minimal fix is a typed envelope: one registered name plus arguments that satisfy that tool's schema.

## 2 - Typed Tool Calls and Fail-Closed Validation

A contract must reject missing quantities, non-positive quantities, and unknown SKUs before execution. The validator is deterministic code; the model does not grade its own proposal.

```mermaid
flowchart LR
    P["Proposed call"] --> N{"Registered name?"}
    N -->|"No"| R["Typed error"]
    N -->|"Yes"| S{"Arguments valid?"}
    S -->|"No"| R
    S -->|"Yes"| E["Execute"]
    style P fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style N fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Build the contract before the registry hides it -----------------------
class InventoryArguments(BaseModel):
    sku: str = Field(pattern=r'^SKU-[A-Z]+-\d{2}$')
    quantity: int = Field(gt=0, le=500)

    @field_validator('sku')
    @classmethod
    def sku_must_exist(cls, value: str) -> str:
        if value not in INVENTORY:
            raise ValueError('unknown_sku')
        return value

class ToolCall(BaseModel):
    name: str
    arguments: dict[str, Any]

class ToolError(BaseModel):
    error_type: str
    message: str
    retryable: bool = False

def validate_inventory_call(proposal: dict[str, Any]) -> InventoryArguments:
    envelope = ToolCall.model_validate(proposal)
    if envelope.name != 'check_inventory':
        raise ValueError(f'unregistered_tool:{envelope.name}')
    return InventoryArguments.model_validate(envelope.arguments)

print('Contract ready: name validation and argument validation are separate gates.')

In [ ]:
# -- Prove the typed contract on the same ten requests ---------------------
typed_results = []
for request in requests:
    proposal = model.propose_inventory_call(request['email'])
    arguments = validate_inventory_call(proposal)
    typed_results.append(arguments.sku == request['sku'] and arguments.quantity == request['quantity'])

typed_rate = sum(typed_results) / len(typed_results)
print(f'Typed valid-call rate: {sum(typed_results)}/{len(typed_results)} ({typed_rate:.0%})')
print(f'Improvement over naive parsing: {typed_rate - naive_rate:+.0%}')
assert typed_results == [True] * 10

unknown = model.propose_inventory_call('Order 5 SKU-UNKNOWN-99 units.')
try:
    validate_inventory_call(unknown)
    raise AssertionError('unknown SKU should not execute')
except ValidationError as error:
    print(f'PASS: unknown SKU failed closed before execution ({error.errors()[0]["msg"]}).')

**Checkpoint:** the same deterministic model improved from an incomplete naive-call rate to 10/10 valid calls because validation moved into software. The model did not get smarter; the system became stricter.

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Let the model invent a tool name | A plausible name can bypass no registry because it resolves to nothing |
| Wrong | Catch every exception as one string | The controller cannot distinguish retryable timeout from invalid input |
| Right | Validate envelope, arguments, then policy | Each failure stops at the boundary that owns it |

## 3 - A Minimal Model-Tool-Observation Loop

A single tool call is useful, but an agent is a loop: inspect state, choose one action, execute it, record the observation, then decide whether to continue. Keep that loop visible before adopting a framework.

```mermaid
flowchart LR
    S["State"] --> M["Choose action"]
    M --> V["Validate"]
    V --> T["Execute tool"]
    T --> O["Observation"]
    O --> S
    style S fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style V fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style T fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Register tools with schemas and deterministic implementations ---------
class ParseArguments(BaseModel):
    email: str = Field(min_length=12)

class QuoteArguments(BaseModel):
    sku: str

def parse_purchase_order(email: str) -> dict[str, Any]:
    proposal = model.propose_inventory_call(email)
    arguments = InventoryArguments.model_validate(proposal['arguments'])
    return {'sku': arguments.sku, 'quantity': arguments.quantity}

def check_inventory(sku: str, quantity: int) -> dict[str, Any]:
    item = INVENTORY[sku]
    return {'sku': sku, 'requested': quantity, 'available': item['available'], 'in_stock': item['available'] >= quantity}

def quote_price(sku: str) -> dict[str, Any]:
    from shared import SUPPLIER_QUOTES
    trusted = [quote for quote in SUPPLIER_QUOTES[sku] if quote['trusted'] and quote['age_hours'] <= 48]
    if not trusted:
        raise ValueError('no_fresh_trusted_quote')
    best = min(trusted, key=lambda quote: quote['unit_price'])
    return {'sku': sku, 'supplier': best['supplier'], 'unit_price': best['unit_price'], 'source_age_hours': best['age_hours']}

TOOLS: dict[str, tuple[type[BaseModel], Callable[..., dict[str, Any]]]] = {
    'parse_purchase_order': (ParseArguments, parse_purchase_order),
    'check_inventory': (InventoryArguments, check_inventory),
    'quote_price': (QuoteArguments, quote_price),
}
print('Registered tools:', ', '.join(TOOLS))

In [ ]:
# -- Execute the visible loop for PO-7293 -------------------------------
def execute_registered(call: dict[str, Any]) -> dict[str, Any]:
    name = call.get('name')
    if name not in TOOLS:
        return ToolError(error_type='unregistered_tool', message=str(name)).model_dump()
    schema, function = TOOLS[name]
    try:
        arguments = schema.model_validate(call.get('arguments', {}))
        return {'ok': True, 'tool': name, 'observation': function(**arguments.model_dump())}
    except (ValidationError, ValueError, KeyError) as error:
        return ToolError(error_type='validation_error', message=str(error)).model_dump()

state = {'request_id': 'PO-7293', 'email': requests[0]['email'], 'trace': [], 'terminal': False}
for step in range(1, 6):
    call = model.choose_action(state)
    if call['name'] == 'finish':
        state['terminal'] = True
        break
    result = execute_registered(call)
    state['trace'].append({'step': step, 'call': call, 'result': result})
    if not result.get('ok'):
        state['terminal'] = True
        state['error'] = result
        break
    observation = result['observation']
    if call['name'] == 'parse_purchase_order':
        state.update(observation, parsed=True)
    elif call['name'] == 'check_inventory':
        state.update(inventory_checked=True, inventory=observation)
    elif call['name'] == 'quote_price':
        state.update(quote_checked=True, quote=observation)

print(json.dumps(state['trace'], indent=2))
assert state['terminal'] and len(state['trace']) == 3
assert [entry['call']['name'] for entry in state['trace']] == ['parse_purchase_order', 'check_inventory', 'quote_price']
print('PASS: PO-7293 reached a terminal state through three attributable tool observations.')

## 4 - Timeouts, Typed Errors, and Idempotent Surfaces

A contract can be valid and still fail at runtime. A timeout should be retryable; an unknown SKU should not. Financial side effects also need an idempotency key so replay cannot create a second commitment. Notebook 09 will build the complete recovery system; here you establish the tool-surface contract.

```mermaid
flowchart TD
    C["Valid call"] --> X{"Outcome"]
    X -->|"Timeout"| T["Retryable typed error"]
    X -->|"Invalid input"| F["Fail closed"]
    X -->|"Side effect"| I["Idempotency key"]
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style T fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Quick health check: boundary failures stay distinguishable ----------
health_checks = {
    'unknown_tool': execute_registered({'name': 'invented_tool', 'arguments': {}}),
    'missing_quantity': execute_registered({'name': 'check_inventory', 'arguments': {'sku': 'SKU-CPU-01'}}),
    'invalid_quantity': execute_registered({'name': 'check_inventory', 'arguments': {'sku': 'SKU-CPU-01', 'quantity': -1}}),
}

assert health_checks['unknown_tool']['error_type'] == 'unregistered_tool'
assert health_checks['missing_quantity']['error_type'] == 'validation_error'
assert health_checks['invalid_quantity']['error_type'] == 'validation_error'
print(json.dumps(health_checks, indent=2))
print('PASS: each bad call failed before tool execution with a typed boundary error.')

**Your turn:** change `TEST_QUANTITY` from `2` to `0`. The printed check should change from `PASS` to `REJECTED` without editing the tool implementation.

In [ ]:
# -- Your turn: change one argument ---------------------------------------
TEST_QUANTITY = 2  # CHANGE THIS: try 0
exercise_call = {'name': 'check_inventory', 'arguments': {'sku': 'SKU-CPU-01', 'quantity': TEST_QUANTITY}}
exercise_result = execute_registered(exercise_call)
print('PASS' if exercise_result.get('ok') else 'REJECTED', exercise_result)

## From Tools to Skills to MCP Servers

These terms describe different layers and should not be used interchangeably:

| Layer | What it contributes | What it does not do |
|---|---|---|
| **Tool** | One executable action with a name, typed inputs, and a result | Decide when a larger task needs that action |
| **Skill** | Reusable task know-how: instructions, examples, checks, and guidance for combining tools | Grant new authority or become an executable service |
| **MCP server** | A protocol endpoint that publishes discoverable tools, resources, or prompts | Decide that a discovered capability is safe to invoke |

A skill can teach an agent how to reconcile a purchase order and can reference inventory or pricing tools. The tools still perform the actions, and deterministic policy still decides whether those actions are allowed. An MCP server can expose those tools across a process or ownership boundary; Chapter 07 builds that lifecycle.

## Roadmap Checkpoint

| Constraint | Before | After |
|---|---:|---:|
| Valid tool calls on ten solvable fixtures | Measured naive rate above | 10/10 |
| Unknown SKU behavior | Could be named in prose | Rejected before execution |
| Action attribution | No executable trace | Tool name, arguments, and observation per step |

```mermaid
flowchart LR
    A["Typed tools unlocked"] --> B["Next failure: loops repeat actions"]
    B --> C["Notebook 01: bounded control"]
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Coverage Ledger

| Tier | Covered here |
|---|---|
| Built and measured | Typed calls, registry, validation, observations, minimal loop, typed errors |
| Explained and illustrated | Skills, MCP-server boundary, timeouts, and idempotent tool surfaces |
| Named with a reason | Framework agents, deferred until the mechanism is visible |

### Key Takeaways

- A tool mention is text; a validated registered call is an action.
- A skill packages reusable know-how; it does not bypass tool validation or policy.
- The model proposes. Deterministic software validates and authorizes.
- Typed errors are control-flow inputs, not prettier exception messages.
- Keep the loop visible until you can explain every transition.